<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/week3b_geo_dna_portrait.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ═══════════════════════════════════════════════════════
#  ЯЧЕЙКА 0. Подготовка данных (из week2b_read_csv.ipynb)
#  Запускать первой в каждом ноутбуке задания 3
# ═══════════════════════════════════════════════════════

# --- Параметры (изменять здесь) ----------------------
RADIUS_KM     = 300   # радиус соседства гор (для week3a)
TOP_N_ROCKS   = 10    # сколько топ-пород использовать
TOP_N_COMPLEX = 20    # сколько самых «сложных» гор брать
# -----------------------------------------------------

import os, pandas as pd, numpy as np
from itertools import combinations

# 1. Клонируем репозиторий (если ещё нет)
repo = "python-ai-AnastasiaKalyashova"
repo_path = f"/content/{repo}"
if not os.path.exists(repo_path):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git
if os.getcwd() != repo_path:
    %cd {repo_path}

# 2. Читаем CSV
file_path = None
for root, dirs, files in os.walk("."):
    if "mountains.csv" in files:
        file_path = os.path.join(root, "mountains.csv")
        break
df = pd.read_csv(file_path)

# 3. Переименование столбцов
if "mountainLabel" in df.columns:
    df = df.rename(columns={
        "mountain":          "URL",
        "mountainLabel":     "mountain",
        "rockMaterialLabel": "rockMaterial",
        "elevationMeters":   "elevation",
    })

# 4. Нормализуем породы
df["rockMaterial"] = df["rockMaterial"].str.lower().str.strip()

# 🔧 ИСПРАВЛЕНИЕ: заменяем "lutite" на "пелит"
df["rockMaterial"] = df["rockMaterial"].replace("lutite", "пелит")

# 5. Парсим координаты
coords = df["coordinates"].str.extract(r'Point\(([^\s]+)\s+([^\s]+)\)')
df["lon"] = pd.to_numeric(coords[0], errors="coerce")
df["lat"] = pd.to_numeric(coords[1], errors="coerce")

# 6. df_unique — по одной строке на гору
df_unique = (
    df.groupby("URL")
    .agg(
        mountain   = ("mountain",     "first"),
        lon        = ("lon",          "first"),
        lat        = ("lat",          "first"),
        elevation  = ("elevation",    "first"),
        rock_count = ("rockMaterial", "nunique"),
        rocks      = ("rockMaterial", lambda x: list(x.unique())),
    )
    .reset_index()
)

# 7. df_clean — только физически возможные высоты
df_clean = df_unique[
    (df_unique.elevation >= 0) &
    (df_unique.elevation <= 8849)
].copy()

# 8. Топ пород по частоте (по df_clean)
top_rocks = (
    df[df["URL"].isin(df_clean["URL"])]
    ["rockMaterial"].value_counts()
    .head(TOP_N_ROCKS).index.tolist()
)

# 9. Co-occurrence матрица пород
pairs = []
for rocks in df_clean["rocks"]:
    clean = [r for r in rocks if r in top_rocks]
    pairs += list(combinations(sorted(set(clean)), 2))
cooc = (pd.DataFrame(pairs, columns=["r1", "r2"])
        .value_counts()
        .reset_index(name="count"))

print(f"✅ Длинный формат:    {len(df)} строк")
print(f"✅ Уникальных гор:    {len(df_unique)}")
print(f"✅ df_clean:          {len(df_clean)} гор (0–8849 м)")
print(f"✅ Топ-{TOP_N_ROCKS} пород:    {top_rocks}")
print(f"✅ Пар co-occurrence: {len(cooc)}")

✅ Длинный формат:    4431 строк
✅ Уникальных гор:    2915
✅ df_clean:          2914 гор (0–8849 м)
✅ Топ-10 пород:    ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']
✅ Пар co-occurrence: 15


In [13]:
# ═══════════════════════════════════════════════════════
# week3b_geo_dna_portrait.ipynb
# Геологический ДНК-портрет гор (исправлены отступы)
# ═══════════════════════════════════════════════════════

import plotly.graph_objects as go
import plotly.express as px

# --- Фильтруем горы с 3 и более породами ---
MIN_ROCKS = 3

filtered_mountains = df_clean[df_clean['rock_count'] >= MIN_ROCKS].copy()
filtered_mountains = filtered_mountains.sort_values('elevation', ascending=False).reset_index(drop=True)

print(f"📊 Гор с {MIN_ROCKS}+ породами: {len(filtered_mountains)}")

# 1. Определяем "высокогорные породы"
high_altitude_rocks = set()
for _, row in filtered_mountains.iterrows():
    if row['elevation'] > 4000:
        high_altitude_rocks.update(row['rocks'])

low_altitude_rocks = set()
for _, row in filtered_mountains.iterrows():
    if row['elevation'] <= 4000:
        low_altitude_rocks.update(row['rocks'])

exclusive_high_rocks = high_altitude_rocks - low_altitude_rocks
print(f"🏔️ Породы, уникальные для высоких гор (>4000 м): {sorted(exclusive_high_rocks)}")

# 2. Получаем все уникальные породы
all_rocks = set()
for rocks in filtered_mountains['rocks']:
    all_rocks.update(rocks)
all_rocks = sorted(all_rocks)

# 3. Создаём цветовую карту
colors = px.colors.qualitative.Plotly + px.colors.qualitative.Set2 + px.colors.qualitative.Set3
rock_color_map = {rock: colors[i % len(colors)] for i, rock in enumerate(all_rocks)}

# 4. Создаём кастомный hover текст
mountain_hover_texts = []
for idx, row in filtered_mountains.iterrows():
    rock_list = ', '.join(row['rocks'])
    star_rocks = [r for r in row['rocks'] if r in exclusive_high_rocks]
    star_text = f"⭐ {', '.join(star_rocks)}" if star_rocks else "нет"

    hover_text = (f"<b>{row['mountain']}</b><br>" +
                  f"📊 Высота: {row['elevation']:.0f} м<br>" +
                  f"🪨 Количество пород: {row['rock_count']}<br>" +
                  f"📋 Состав: {rock_list}<br>" +
                  f"⭐ Уникальные высокогорные породы: {star_text}")
    mountain_hover_texts.append(hover_text)

# 5. Строим график
fig = go.Figure()

# Добавляем породы
for rock in all_rocks:
    rock_presence = [1 if rock in row['rocks'] else 0 for _, row in filtered_mountains.iterrows()]
    display_name = f"{rock} ⭐" if rock in exclusive_high_rocks else rock

    fig.add_trace(go.Bar(
        x=filtered_mountains['mountain'],
        y=rock_presence,
        name=display_name,
        orientation='v',
        marker_color=rock_color_map[rock],
        legendgroup=rock,
        hoverinfo='none',
        showlegend=True
    ))

# 6. Точки высоты
fig.add_trace(go.Scatter(
    x=filtered_mountains['mountain'],
    y=filtered_mountains['elevation'],
    mode='markers',
    name='Высота горы (м)',
    marker=dict(
        size=12,
        color='black',
        symbol='circle',
        line=dict(width=2, color='white')
    ),
    yaxis='y2',
    hoverinfo='skip',
    showlegend=True
))

# 7. Невидимые столбцы для hover
fig.add_trace(go.Bar(
    x=filtered_mountains['mountain'],
    y=[max(filtered_mountains['rock_count']) + 0.5] * len(filtered_mountains),
    name='Информация о горе',
    orientation='v',
    marker_color='rgba(0,0,0,0)',
    hoverinfo='text',
    hovertext=mountain_hover_texts,
    showlegend=False,
    hovertemplate='%{hovertext}<extra></extra>'
))

# 8. Настройка layout - ИСПРАВЛЕННЫЕ ОТСТУПЫ
fig.update_layout(
    title=dict(
        text=f"🏔️ Геологический ДНК-портрет: Горы с {MIN_ROCKS}+ породами ({len(filtered_mountains)} шт)<br>"
             f"<sup>⭐ — породы, встречающиеся ТОЛЬКО у гор выше 4000 м | Наведите на столбец для деталей</sup>",
        font=dict(size=16),
        x=0.5
    ),
    barmode='stack',
    height=600,
    width=max(1200, len(filtered_mountains) * 40),
    showlegend=True,
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=10)
    ),
    hovermode='x unified',

    # Ось X - убираем отступы
    xaxis=dict(
        title='Горы (от самых высоких к низким)',
        titlefont=dict(size=12),
        tickfont=dict(size=10),
        tickangle=-45,
        categoryorder='array',
        categoryarray=filtered_mountains['mountain'].tolist(),
        # Убираем отступы по краям
        automargin=True,
        showline=True,
        linecolor='black',
        linewidth=1
    ),

    # Первая ось Y - начинается строго от 0
    yaxis=dict(
        title='Количество пород',
        titlefont=dict(size=12),
        range=[0, filtered_mountains['rock_count'].max() + 0.5],  # строго от 0
        side='left',
        gridcolor='lightgray',
        dtick=1,
        tickformat='d',
        domain=[0, 0.7],
        showline=True,
        linecolor='black',
        linewidth=1,
        zeroline=True,  # показываем линию y=0
        zerolinecolor='black',
        zerolinewidth=1
    ),

    # Вторая ось Y
    yaxis2=dict(
        title='Высота над уровнем моря (м)',
        titlefont=dict(size=12, color='black'),
        tickfont=dict(color='black', size=10),
        overlaying='y',
        side='right',
        range=[0, filtered_mountains['elevation'].max() * 1.05],
        showgrid=False,
        position=0.85,
        showline=True,
        linecolor='black',
        linewidth=1
    ),

    margin=dict(l=80, r=120, t=100, b=200),
    plot_bgcolor='white',
    paper_bgcolor='white',
    bargap=0.2,  # промежуток между столбцами
    bargroupgap=0.1  # промежуток внутри группы
)

# 9. Линия 4000 м
fig.add_shape(
    type="line",
    x0=-0.5,
    x1=len(filtered_mountains) - 0.5,
    y0=4000,
    y1=4000,
    xref="x",
    yref="y2",
    line=dict(color="red", width=2, dash="dash"),
    opacity=0.6
)

fig.add_annotation(
    x=len(filtered_mountains) - 1,
    y=4000,
    xref="x",
    yref="y2",
    text="🏔️ 4000 м",
    showarrow=False,
    font=dict(size=10, color="red"),
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="red",
    borderwidth=1,
    borderpad=3,
    xanchor='right',
    yanchor='bottom'
)

# Принудительно устанавливаем диапазон X
fig.update_xaxes(range=[-0.5, len(filtered_mountains) - 0.5])

fig.show()

# 10. Статистика
print("\n" + "="*70)
print("📊 СТАТИСТИКА")
print("="*70)

correlation = filtered_mountains['rock_count'].corr(filtered_mountains['elevation'])
print(f"📈 Корреляция: {correlation:.3f}")

print(f"\n📊 Распределение по количеству пород:")
for i in range(3, filtered_mountains['rock_count'].max() + 1):
    count = (filtered_mountains['rock_count'] == i).sum()
    print(f"   {i} породы(од): {count} гор")

print(f"\n🏔️ Самая высокая гора: {filtered_mountains.iloc[0]['mountain']} ({filtered_mountains.iloc[0]['elevation']:.0f} м)")
print(f"🏔️ Самая низкая гора: {filtered_mountains.iloc[-1]['mountain']} ({filtered_mountains.iloc[-1]['elevation']:.0f} м)")

📊 Гор с 3+ породами: 348
🏔️ Породы, уникальные для высоких гор (>4000 м): ['вулканический туф']



📊 СТАТИСТИКА
📈 Корреляция: 0.305

📊 Распределение по количеству пород:
   3 породы(од): 336 гор
   4 породы(од): 9 гор
   5 породы(од): 3 гор

🏔️ Самая высокая гора: Эльбрус (5642 м)
🏔️ Самая низкая гора: Puig de s'Alqueria (52 м)
